# Unconstrained seller value

This follow-up to slide 11 compares the 40% minimum with maximizing fitted seller value without a conversion requirement. It reuses the seeds, model specifications, 200-vehicle portfolios and candidate floor grid of the 30 binding-price, historical-floor replications. The simulator's response functions score both policies and provide the unconstrained oracle benchmark. Expected conversion is the simulator's expected share sold. Seller value is averaged over all vehicles offered.

In [1]:
import nbformat
from threadpoolctl import threadpool_limits

# Reuse setup and function definitions without running the original experiments.
study = nbformat.read('05_single_run_policy_study.ipynb', as_version=4)
for index in (1, 2, 4, 5, 7):
    code = study.cells[index].source
    if index == 1:
        code = '\n'.join(line for line in code.splitlines() if not line.startswith('OUT='))
    exec(compile(code, f'05_single_run_policy_study.ipynb:cell_{index}', 'exec'))


In [ ]:
def compare_floors(replication):
    seed = 50000 + replication
    institution, assignment = 'binding', 'historical'
    simulated, _ = generate(seed, institution, assignment, N)
    train, validation, test = split_dates(simulated)
    model = refit(train, validation, institution)
    portfolio = test.sample(PORTFOLIO, random_state=seed).reset_index(drop=True)
    grid, _ = candidates(portfolio, support_fit(train), 1.0, 0.0)
    expanded = portfolio.loc[portfolio.index.repeat(grid.shape[1])].copy()
    expanded['floor_ratio'] = grid.ravel()
    q, price = (item.reshape(grid.shape) for item in predict(model, expanded))
    fitted_value = value(q, q * price, portfolio)
    oracle_q, oracle_proceeds = oracle(portfolio, grid, assignment, institution)
    oracle_value = value(oracle_q, oracle_proceeds, portfolio)

    # Without a portfolio-wide minimum, maximize separately for each vehicle.
    unconstrained = fitted_value.argmax(axis=1)
    constrained, _ = optimize(q, fitted_value, .40)
    choices = {
        '40% minimum': constrained,
        'No minimum': unconstrained,
        'Oracle, no minimum': oracle_value.argmax(axis=1),
    }
    rows = np.arange(len(portfolio))
    return [
        {'replication': replication, 'policy': policy,
             'predicted_conversion': q[rows, chosen].mean(),
             'expected_conversion': oracle_q[rows, chosen].mean(),
             'expected_value': oracle_value[rows, chosen].mean()}
        for policy, chosen in choices.items()
    ]


In [3]:
with threadpool_limits(limits=1):
    results = pd.DataFrame(
        row for rep in range(REPLICATIONS) for row in compare_floors(rep)
    )
original = pd.read_csv('data/study/all_policy_results.csv')
reference = original.query("scenario == 'binding_historical' and target == 0.4 and policy == 'optimized'")
metrics = ['expected_value', 'expected_conversion']
matched = results.query("policy == '40% minimum'").merge(
    reference[['replication', *metrics]], on='replication',
    suffixes=('_new', '_saved'), validate='one_to_one'
)
assert len(matched) == REPLICATIONS
for metric in metrics:
    np.testing.assert_allclose(matched[f'{metric}_new'], matched[f'{metric}_saved'], rtol=1e-5, atol=1e-6)
summary = results.groupby('policy', sort=False)[metrics].mean()
print(summary.to_string(formatters={
    'expected_value': lambda value: f'${value:,.2f}',
    'expected_conversion': lambda rate: f'{rate:.2%}',
}))


                   expected_value expected_conversion
policy                                               
40% minimum             $8,465.66              44.67%
No minimum              $8,479.11              40.42%
Oracle, no minimum      $8,499.38              40.85%
